# SGA Workshop

Energy demand forecasting — work through all sections top to bottom.

The first cell sets up your environment automatically.

In [ ]:
import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

DEPLOY_REPO_URL = "https://github.com/RichardPovinelli/SGA_workshop_deploy_test.git"
DEPLOY_REPO_DIR = "SGA_workshop_deploy_test"

# Clone deploy repo if not already in it (e.g. opened directly from GitHub in Colab)
if not Path("src").exists():
    if not Path(DEPLOY_REPO_DIR).exists():
        subprocess.check_call(["git", "clone", DEPLOY_REPO_URL, "-q"])
    os.chdir(DEPLOY_REPO_DIR)

# Install requirements if needed
pandas_spec = importlib.util.find_spec("pandas")
xgboost_spec = importlib.util.find_spec("xgboost")
if pandas_spec is None or xgboost_spec is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"])

# Make src/ importable
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Install dataset if not already installed
src_runtime_assets = importlib.import_module("src.runtime_assets")
src_config = importlib.import_module("src.config")

load_install_state = src_runtime_assets.load_install_state
install_dataset_from_file = src_runtime_assets.install_dataset_from_file
get_dataset_asset_descriptor = src_runtime_assets.get_dataset_asset_descriptor
DATASET_INSTALL_STATE_PATH = src_config.DATASET_INSTALL_STATE_PATH
DATASET_MANIFEST_FILENAME = src_config.DATASET_MANIFEST_FILENAME

if load_install_state(DATASET_INSTALL_STATE_PATH) is None:
    descriptor = get_dataset_asset_descriptor()
    install_dataset_from_file(
        Path("data") / descriptor.filename,
        source_manifest_path=Path("data") / DATASET_MANIFEST_FILENAME,
        requested_release_tag=descriptor.release_tag,
        requested_filename=descriptor.filename,
    )
    print("Dataset installed.")
else:
    print("Environment ready.")

---
## Part 1: Dataset Setup

Load the installed dataset and inspect the workshop-ready schema.

In [ ]:
from src.data.load import load_dataset

df = load_dataset()
df.head()

In [ ]:
{
    "rows": len(df),
    "columns": list(df.columns),
    "zones": sorted(df["zone"].unique().tolist()),
    "split_counts": df["split"].value_counts().sort_index().to_dict(),
}

---
## Part 2: Baseline Models

Run the baseline models for one selected horizon and compare metrics.

In [ ]:
import pandas as pd

from src.data.split import get_test_df
from src.evaluation.diagnostics import ljung_box_diagnostics
from src.evaluation.dm_test import diebold_mariano_test
from src.models.lasso import evaluate_lasso_model, predict_lasso
from src.models.linear import evaluate_linear_model, predict_linear
from src.models.naive import evaluate_naive_model, predict_naive

test_df = get_test_df(df)
horizon = 1

naive_results = evaluate_naive_model(test_df, horizon)
linear_model, linear_results = evaluate_linear_model(df, horizon)
lasso_model, lasso_results = evaluate_lasso_model(df, horizon)

linear_predictions = predict_linear(linear_model, test_df)
lasso_predictions = predict_lasso(lasso_model, test_df)
naive_predictions = predict_naive(test_df, horizon)

comparison = pd.concat([naive_results, linear_results, lasso_results], ignore_index=True)
comparison

In [ ]:
dm_result = diebold_mariano_test(test_df["target_h1"], linear_predictions, naive_predictions)
diagnostics = ljung_box_diagnostics(test_df["target_h1"] - linear_predictions, lags=[5])
{
    "dm_result": dm_result,
    "ljung_box": diagnostics.to_dict(orient="records"),
}

---
## Part 3: XGBoost

Train the workshop XGBoost model and compare it to the linear baselines.

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.models.xgb import evaluate_xgb_for_all_zones

_, xgb_results = evaluate_xgb_for_all_zones(df, horizon)
_, linear_results = evaluate_linear_model(df, horizon)
results = xgb_results[["zone", "rmse"]].merge(
    linear_results[["zone", "rmse"]],
    on="zone",
    suffixes=("_xgb", "_linear"),
)
results

In [ ]:
ax = results.set_index("zone").plot(kind="bar", figsize=(8, 4), title="Per-zone RMSE")
ax.set_ylabel("RMSE")
plt.tight_layout()

---
## Part 4: Multi-Horizon Forecasting

Run direct XGBoost across horizons 1 through 7 and inspect degradation.

In [ ]:
all_results = []
for h in range(1, 8):
    _, horizon_results = evaluate_xgb_for_all_zones(df, h)
    all_results.append(horizon_results)

multi_results = pd.concat(all_results, ignore_index=True)
multi_results

In [ ]:
summary = multi_results.groupby("horizon", as_index=False)["rmse"].mean()
ax = summary.plot(x="horizon", y="rmse", marker="o", title="Average RMSE by Horizon")
ax.set_ylabel("RMSE")
plt.tight_layout()

---
## Part 5: TFT Inference

Load the installed TFT checkpoints, run inference for one zone, and compare to XGBoost.

In [ ]:
from src.data.split import get_test_df
from src.evaluation.dm_test import diebold_mariano_test
from src.models.tft_inference import evaluate_tft_model, predict_tft
from src.models.xgb import fit_xgb_for_zone, predict_xgb

zone = sorted(df["zone"].unique().tolist())[0]
zone_df = df.loc[df["zone"] == zone]
zone_test_df = get_test_df(zone_df)

xgb_model = fit_xgb_for_zone(df, horizon, zone)
xgb_predictions = predict_xgb(xgb_model, zone_test_df)
tft_predictions = predict_tft(zone_test_df, horizon)
tft_results = evaluate_tft_model(zone_test_df, horizon)
tft_results

In [ ]:
diebold_mariano_test(zone_test_df["target_h1"], tft_predictions, xgb_predictions)